# きのこ・たけのこの例

上武康亮・遠山祐太・若森直樹・渡辺安虎/著『実証ビジネス・エコノミクス』（日本評論社、2025年12月刊）のサンプルデータ

[empirical_business_economics/01_Discrete_Choice_Ch02 at main · keisemi/empirical_business_economics](https://github.com/keisemi/empirical_business_economics/tree/main/01_Discrete_Choice_Ch02)




In [15]:
import pandas as pd

# アンケート個票データをDL
DATA_URL="https://raw.githubusercontent.com/keisemi/empirical_business_economics/refs/heads/main/01_Discrete_Choice_Ch02/data/KinokoTakenokoSurvey_raw.csv"
df = pd.read_csv(DATA_URL)

display(df.head(3))

# 列名を変更
new_col_names = ["ID", "experience", "Q1", "Q2", "Q3", "Q4", "Q5", "age", "gender", "region", "familyhouse"]
df = df.iloc[:, 7:17].copy()
df.insert(0, "ID", range(1, len(df) + 1))
df.columns = new_col_names

# 対象外レコードを削除
df = df[
    (df["experience"] != "4 : 食べたことがない") & (df["gender"] != "3 : 回答したくない")
].dropna()
df = df.reset_index(drop=True)

,回答,送信完了：,コース,グループ,ID,フルネーム,管理者用ユーザ,Q00_経験調査,"Q00_Q2_if_(200,200)","Q00_Q2_if_(180,200)","Q00_Q2_if_(200,170)","Q00_Q2_if_(220,200)","Q00_Q2_if_(190,210)",Q00_age,Q00_sex,Q00_area,Q00_familyhouse,Q00_consensus
0,2730361,2024/04/19 16:57:17,産業組織論 ０１,NaN,NaN,匿名1,NaN,3 : １年以上前,2 : たけのこの里を買う,1 : きのこの山を買う,2 : たけのこの里を買う,2 : たけのこの里を買う,1 : きのこの山を買う,25.0,1 : 男性,3 : 関東地方,1.0,1
1,2742697,2024/04/23 13:17:19,産業組織論 ０１,NaN,NaN,匿名2,NaN,3 : １年以上前,1 : きのこの山を買う,1 : きのこの山を買う,2 : たけのこの里を買う,2 : たけのこの里を買う,1 : きのこの山を買う,27.0,1 : 男性,9 : 海外,0.0,1
2,2728507,2024/04/19 12:42:50,産業組織論 ０１,NaN,NaN,匿名3,NaN,1 : 過去半年以内,2 : たけのこの里を買う,2 : たけのこの里を買う,1 : きのこの山を買う,2 : たけのこの里を買う,2 : たけのこの里を買う,23.0,1 : 男性,3 : 関東地方,1.0,1


In [56]:
# 縦持ちへ変換
q_cols = [c for c in df.columns if c.startswith("Q")]
id_cols = [c for c in df.columns if c not in q_cols]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=q_cols,
    var_name="occasion",
    value_name="choice",
)

choice_map = {
    "1 : きのこの山を買う": 1,
    "2 : たけのこの里を買う": 2,
    "3 : どちらも買わない": 0,
}

df_long["choice"] = df_long["choice"].map(choice_map)

# 各選択肢での価格を設定
price_df = pd.DataFrame(
    {
        "occasion": ["Q1", "Q2", "Q3", "Q4", "Q5"],
        "price_0": [0, 0, 0, 0, 0],
        "price_1": [200, 180, 200, 220, 190],
        "price_2": [200, 200, 170, 200, 210],
    }
)
df_long = df_long.merge(price_df, on="occasion")

# どの選択肢を選んだかダミーにする場合
dummies = pd.get_dummies(df_long["choice"], prefix="choice").astype("Int8")
df_long = pd.concat([df_long, dummies], axis=1)

# 使うカラムだけ選ぶ
df_long = df_long.filter(regex="choice|price")
df_long.tail(3)

,choice,price_0,price_1,price_2,choice_0,choice_1,choice_2
1177,1,0,190,210,0,1,0
1178,1,0,190,210,0,1,0
1179,1,0,190,210,0,1,0


In [57]:
print(df_long)

      choice  price_0  price_1  price_2  choice_0  choice_1  choice_2
0          2        0      200      200         0         0         1
1          1        0      200      200         0         1         0
2          2        0      200      200         0         0         1
3          1        0      200      200         0         1         0
4          2        0      200      200         0         0         1
...      ...      ...      ...      ...       ...       ...       ...
1175       1        0      190      210         0         1         0
1176       1        0      190      210         0         1         0
1177       1        0      190      210         0         1         0
1178       1        0      190      210         0         1         0
1179       1        0      190      210         0         1         0

[1180 rows x 7 columns]


### 多項ロジットモデル

選択肢は3つ：

0. 買わない（outside goods）
1. きのこ
2. たけのこ



選択肢 $j \in \mathcal{J} \equiv\{$ Kinoko，Takenoko，outside $\}$ から得られる効用 $U_{i, k, j}$ を以下のように与える。

$$
\begin{aligned}
U_{i, k, \text { Kinoko }} & =\beta_{\text {Kinoko }}-\alpha p_{\text {Kinoko }, k}+\epsilon_{i, k, \text { Kinoko }} \\
U_{i, k, \text { Takenoko }} & =\beta_{\text {Takenoko }}-\alpha p_{\text {Takenoko }, k}+\epsilon_{i, k, \text { Takenoko }} \\
U_{i, k, \text { outside }} & =\epsilon_{i, k, \text { outside }}
\end{aligned}
$$

ここで、
- $p_{j, k}$ は設問 $k$ における選択肢 $j$ の価格
- $\epsilon_{i, j, k}$ は i．i．d．の第 I 種極値分布に従う選好ショック

$$
\begin{aligned}
& P_k(\text { Kinoko } \mid \theta) \\
& \qquad=\frac{\exp \left(\beta_{\text {Kinoko }}-\alpha p_{\text {Kinoko }, k}\right)}{1+\exp \left(\beta_{\text {Kinoko }}-\alpha p_{\text {Kinoko }, k}\right)+\exp \left(\beta_{\text {Takenoko }}-\alpha p_{\text {Takenoko }, k}\right)} \\
& P_k(\text { Takenoko } \mid \theta) \\
& \quad=\frac{\exp \left(\beta_{\text {Takenoko }}-\alpha p_{\text {Takenoko }, k}\right)}{1+\exp \left(\beta_{\text {Kinoko }}-\alpha p_{\text {Kinoko }, k}\right)+\exp \left(\beta_{\text {Takenoko }}-\alpha p_{\text {Takenoko }, k}\right)} \\
& P_k(\text { outside } \mid \theta) \\
& \quad=\frac{1}{1+\exp \left(\beta_{\text {Kinoko }}-\alpha p_{\text {Kinoko }, k}\right)+\exp \left(\beta_{\text {Takenoko }}-\alpha p_{\text {Takenoko }, k}\right)}
\end{aligned}
$$

まとめると

$$
Pr(d_i=j)
= \frac
{\exp( \beta_j x_j - \alpha p_j )}
{1 + \sum^J_{l=1} \exp( \beta_l x_j - \alpha p_l )}
$$

- $p_j$: 財$j$の価格
- $x_j$: 財$j$であることを示す$\{0,1\}$の変数

共通の価格感応度$\alpha$なのが通常のロジットモデルと異なる点（→ conditional logit model）



In [ ]:

# きのこ、たけのこどちらを選んだか、選択肢ごとにフラグを建てる場合
# df_long = (
#     df_long.merge(price_df, on="occasion")
#     .assign(
#         Kinoko_0=0,
#         Kinoko_1=1,
#         Kinoko_2=0,
#         Takenoko_0=0,
#         Takenoko_1=0,
#         Takenoko_2=1,
#     )
#     .sort_values(["ID", "occasion"])
#     .reset_index(drop=True)
# )


# 使うカラムだけ選ぶ
# df_long = df_long.filter(regex="choice|price|Kinoko|Takenoko")

import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm

# 選択肢ごとの説明変数を (N, J=3, K=3) の配列にまとめる
# J: 0=買わない, 1=きのこ, 2=たけのこ / K: price, Kinoko, Takenoko
price = df_long[["price_0", "price_1", "price_2"]].to_numpy(dtype=float)
kinoko = df_long[["Kinoko_0", "Kinoko_1", "Kinoko_2"]].to_numpy(dtype=float)
takenoko = df_long[["Takenoko_0", "Takenoko_1", "Takenoko_2"]].to_numpy(dtype=float)
choice = df_long["choice"].to_numpy(dtype=int)

X = np.stack([price, kinoko, takenoko], axis=2)
N, J, K = X.shape


def choice_prob(beta, X):
    """beta = [price係数(-alpha), beta_Kinoko, beta_Takenoko] から選択確率 P(j) を計算"""
    V = X @ beta  # (N, J) 効用 V_ij = beta・X_ij
    return np.exp(V - logsumexp(V, axis=1, keepdims=True))


def neg_log_likelihood(beta, X, choice):
    V = X @ beta
    log_denom = logsumexp(V, axis=1)
    log_p_chosen = V[np.arange(len(choice)), choice] - log_denom
    return -log_p_chosen.sum()


def gradient(beta, X, choice):
    # d(logL)/d(beta) = sum_i [ X_i,選んだ選択肢 - E_P[X_i] ]
    P = choice_prob(beta, X)
    X_chosen = X[np.arange(len(choice)), choice, :]
    X_expected = np.einsum("nj,njk->nk", P, X)
    return -(X_chosen - X_expected).sum(axis=0)


def hessian(beta, X, choice):
    # d^2(-logL)/d(beta)^2 = sum_i Var_P[X_i] = sum_i (E[XX^T] - E[X]E[X]^T)
    P = choice_prob(beta, X)
    EX = np.einsum("nj,njk->nk", P, X)
    EXX = np.einsum("nj,njk,njl->nkl", P, X, X)
    Var = EXX - np.einsum("nk,nl->nkl", EX, EX)
    return Var.sum(axis=0)

In [36]:
beta0 = np.zeros(K)
result = minimize(
    neg_log_likelihood, beta0, args=(X, choice), jac=gradient, hess=hessian, method="trust-exact"
)

beta_hat = result.x
cov = np.linalg.inv(hessian(beta_hat, X, choice))  # 尤度のヘシアン(観測情報行列)の逆行列
se = np.sqrt(np.diag(cov))
z = beta_hat / se
p_value = 2 * (1 - norm.cdf(np.abs(z)))

summary = pd.DataFrame(
    {"coef": beta_hat, "std err": se, "z": z, "P>|z|": p_value},
    index=["price", "Kinoko", "Takenoko"],
).round(3)

print(f"converged: {result.success}")
print(f"Log-Likelihood: {-result.fun:.3f}")
summary

converged: True
Log-Likelihood: -1067.662


,coef,std err,z,P>|z|
price,-0.057,0.004,-15.304,0.0
Kinoko,11.651,0.729,15.974,0.0
Takenoko,12.202,0.745,16.378,0.0


In [38]:
# --- einsumを使わない、for文で書いた読みやすい版（計算内容は上のセルと同じ） ---


def choice_prob_loop(beta, X):
    """観測ごとに愚直にループして選択確率 P(j) を計算する"""
    N, J, K = X.shape
    P = np.zeros((N, J))
    for i in range(N):
        V_i = np.array([X[i, j, :] @ beta for j in range(J)])  # 選択肢ごとの効用
        V_i = V_i - V_i.max()  # オーバーフロー対策(最大値を引いても確率は変わらない)
        exp_V = np.exp(V_i)
        P[i, :] = exp_V / exp_V.sum()
    return P


def neg_log_likelihood_loop(beta, X, choice):
    P = choice_prob_loop(beta, X)
    N = X.shape[0]
    log_lik = 0.0
    for i in range(N):
        log_lik += np.log(P[i, choice[i]])
    return -log_lik


def gradient_loop(beta, X, choice):
    # d(logL)/d(beta) = sum_i [ 選んだ選択肢のX - 確率で重み付けた期待値のX ]
    N, J, K = X.shape
    P = choice_prob_loop(beta, X)
    grad = np.zeros(K)
    for i in range(N):
        x_chosen = X[i, choice[i], :]
        x_expected = np.zeros(K)
        for j in range(J):
            x_expected += P[i, j] * X[i, j, :]
        grad += x_chosen - x_expected
    return -grad


def hessian_loop(beta, X, choice):
    # d^2(-logL)/d(beta)^2 = sum_i Var_P[X_i]（P で重み付けた分散共分散行列の合計）
    N, J, K = X.shape
    P = choice_prob_loop(beta, X)
    hess = np.zeros((K, K))
    for i in range(N):
        x_expected = np.zeros(K)
        for j in range(J):
            x_expected += P[i, j] * X[i, j, :]
        for j in range(J):
            diff = X[i, j, :] - x_expected
            hess += P[i, j] * np.outer(diff, diff)
    return hess


result_loop = minimize(
    neg_log_likelihood_loop,
    np.zeros(K),
    args=(X, choice),
    jac=gradient_loop,
    hess=hessian_loop,
    method="trust-exact",
)

beta_hat = result_loop.x
cov = np.linalg.inv(hessian(beta_hat, X, choice))  # 尤度のヘシアン(観測情報行列)の逆行列
se = np.sqrt(np.diag(cov))
z = beta_hat / se
p_value = 2 * (1 - norm.cdf(np.abs(z)))

summary = pd.DataFrame(
    {"coef": beta_hat, "std err": se, "z": z, "P>|z|": p_value},
    index=["price", "Kinoko", "Takenoko"],
).round(3)
display(summary)

print(f"converged: {result_loop.success}")
print(f"Log-Likelihood: {-result_loop.fun:.3f}")
print("einsum版と同じ結果か:", np.allclose(result_loop.x, result.x))

,coef,std err,z,P>|z|
price,-0.057,0.004,-15.304,0.0
Kinoko,11.651,0.729,15.974,0.0
Takenoko,12.202,0.745,16.378,0.0


converged: True
Log-Likelihood: -1067.662
einsum版と同じ結果か: True


In [58]:
# --- 「選んだ選択肢」をワンホットダミー(choice_0/1/2)にした版 ---
# fancy indexing(X[np.arange(N), choice])の代わりに、
# 対数尤度 logL = sum_i sum_j y_ij * log(P_ij) をそのまま計算する

price_cols = ["price_0", "price_1", "price_2"]
choice_cols = ["choice_0", "choice_1", "choice_2"]

price = df_long[price_cols].to_numpy(dtype=float)  # (N, J) 選択肢ごとの価格
y = df_long[choice_cols].to_numpy(dtype=float)  # (N, J) 選んだ選択肢が1のワンホットダミー

X = price[:, :, None]  # (N, J, K=1) 説明変数は価格のみ
N, J, K = X.shape


def choice_prob(beta, X):
    V = X @ beta  # (N, J) 効用
    V = V - V.max(axis=1, keepdims=True)  # オーバーフロー対策
    exp_V = np.exp(V)
    return exp_V / exp_V.sum(axis=1, keepdims=True)


def neg_log_likelihood(beta, X, y):
    P = choice_prob(beta, X)
    return -(y * np.log(P)).sum()  # yはワンホットなので、選んだ選択肢の対数確率だけが残る


def gradient(beta, X, y):
    P = choice_prob(beta, X)
    X_chosen = (y[:, :, None] * X).sum(axis=(0, 1))  # 実際に選んだ選択肢のXの合計
    X_expected = (P[:, :, None] * X).sum(axis=(0, 1))  # 確率で重み付けたXの期待値の合計
    return -(X_chosen - X_expected)


def hessian(beta, X, y):
    P = choice_prob(beta, X)
    EX = (P[:, :, None] * X).sum(axis=1)  # (N, K)
    hess = np.zeros((K, K))
    for j in range(J):
        diff = X[:, j, :] - EX  # (N, K)
        weighted = P[:, j, None] * diff  # (N, K)
        hess += (weighted[:, :, None] * diff[:, None, :]).sum(axis=0)
    return hess


beta0 = np.zeros(K)
result = minimize(neg_log_likelihood, beta0, args=(X, y), jac=gradient, hess=hessian, method="trust-exact")

beta_hat = result.x
cov = np.linalg.inv(hessian(beta_hat, X, y))
se = np.sqrt(np.diag(cov))
z = beta_hat / se
p_value = 2 * (1 - norm.cdf(np.abs(z)))

summary = pd.DataFrame({"coef": beta_hat, "std err": se, "z": z, "P>|z|": p_value}, index=["price"]).round(4)

print(f"converged: {result.success}")
print(f"Log-Likelihood: {-result.fun:.3f}")
summary

converged: True
Log-Likelihood: -1261.518


,coef,std err,z,P>|z|
price,0.0028,0.0004,7.9706,0.0
